In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# sys.path.insert(0, "/Users/poorna/Downloads/CE-updated/calibrated_explanations/src")

# Import the necessary libraries
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from calibrated_explanations import WrapCalibratedExplainer, __version__

print(f"calibrated_explanations {__version__}")

calibrated_explanations 0.11.3


In [3]:
# Load and preprocess the data
num_to_test = 10  # number of instances to test, one from each class
dataset = "diabetes_full"
delimiter = ","
model = "RF"

filename = "../../data/diabetes_full.csv"
df = pd.read_csv(filename, delimiter=delimiter)
target = "Y"
X, y = df.drop(target, axis=1), df[target]
no_of_classes = len(np.unique(y))
no_of_features = X.shape[1]
no_of_instances = X.shape[0]

# find categorical features
categorical_features = [i for i in range(no_of_features) if len(np.unique(X.iloc[:, i])) < 10]

# select test instances from each class and split into train, cal and test
idx = np.argsort(y.values).astype(int)
X, y = X.values[idx, :], y.values[idx]
test_index = np.array(
    [
        *range(int(num_to_test / 2)),
        *range(no_of_instances - 1, no_of_instances - int(num_to_test / 2) - 1, -1),
    ]
)
train_index = np.setdiff1d(np.array(range(no_of_instances)), test_index)
x_train, x_test = X[train_index, :], X[test_index, :]
y_train, y_test = y[train_index], y[test_index]
X_prop_train, x_cal, y_prop_train, y_cal = train_test_split(
    x_train, y_train, test_size=0.33, random_state=42, stratify=y_train
)

In [4]:
# Train the model and create the explainer
model = RandomForestClassifier()

model.fit(X_prop_train, y_prop_train)

ce = WrapCalibratedExplainer(model)
ce.calibrate(x_cal, y_cal, feature_names=df.columns, categorical_features=categorical_features, class_labels={0: "Non-diabetic", 1: "Diabetic"})

WrapCalibratedExplainer(learner=RandomForestClassifier(), fitted=True, calibrated=True, 
		explainer=CalibratedExplainer(mode=classification, learner=RandomForestClassifier()))

In [5]:
factual_explanations = ce.explain_factual(x_test)
print("Probability [lower and upper bound] for Diabetic:")
print(
    *zip(
        [
            f"Instance {i}: {exp.prediction['predict']:.3f} [{exp.prediction['low']:5.3f}, {exp.prediction['high']:5.3f}]"
            for i, exp in enumerate(factual_explanations)
        ], strict=False
    ),
    sep="\n",
)

Probability [lower and upper bound] for Diabetic:
('Instance 0: 0.468 [0.457, 0.478]',)
('Instance 1: 0.468 [0.457, 0.478]',)
('Instance 2: 0.468 [0.457, 0.478]',)
('Instance 3: 0.056 [0.000, 0.059]',)
('Instance 4: 0.596 [0.579, 0.622]',)
('Instance 5: 0.386 [0.364, 0.400]',)
('Instance 6: 0.479 [0.457, 0.500]',)
('Instance 7: 0.612 [0.600, 0.630]',)
('Instance 8: 0.612 [0.600, 0.630]',)
('Instance 9: 0.190 [0.171, 0.195]',)


Example demonstrating the new to_narrative API.

This shows the clean API for generating narratives from calibrated explanations.

In [6]:
# The new to_narrative method provides a clean API:
explanations = factual_explanations

# Basic usage
narratives = explanations[0].to_narrative(
    expertise_level=("beginner", "advanced"),
    output_format="text",
)
print(narratives)


Instance 0

Factual Explanation (Advanced):
--------------------------------------------------------------------------------
Prediction: Diabetic
Calibrated Probability: 0.468
Prediction Interval: [0.457, 0.478]

Factors impacting the calibrated probability for class Diabetic positively:
Glucose (145.0) > 139.50  — weight ≈ 0.278 [0.273, 0.297]
Age (57.0) > 27.50        — weight ≈ 0.069 [0.045, 0.103]
Pregnancies (13.0) > 6.50 — weight ≈ 0.050 [0.019, 0.091]

Factors impacting the calibrated probability for class Diabetic negatively:
BMI (22.2) <= 26.90       — weight ≈ -0.138 [-0.159, -0.125]

Factual Explanation (Beginner):
--------------------------------------------------------------------------------
Prediction: Diabetic
Calibrated Probability: 0.468

Factors impacting the probability for class Diabetic positively:
* Glucose (145.0) > 139.50
* Age (57.0) > 27.50
* Pregnancies (13.0) > 6.50

Factors impacting the probability for class Diabetic negatively:
* BMI (22.2) <= 26.90


In [7]:
# Different output formats:

# 1. DataFrame (default) - returns pandas DataFrame
df_narratives = explanations.to_narrative(
    expertise_level=("beginner", "advanced"), output_format="markdown"
)
print(df_narratives)

## Instance 0

### Factual Explanation (Advanced)

```
Prediction: Diabetic
Calibrated Probability: 0.468
Prediction Interval: [0.457, 0.478]

Factors impacting the calibrated probability for class Diabetic positively:
Glucose (145.0) > 139.50  — weight ≈ 0.278 [0.273, 0.297]
Age (57.0) > 27.50        — weight ≈ 0.069 [0.045, 0.103]
Pregnancies (13.0) > 6.50 — weight ≈ 0.050 [0.019, 0.091]

Factors impacting the calibrated probability for class Diabetic negatively:
BMI (22.2) <= 26.90       — weight ≈ -0.138 [-0.159, -0.125]
```

### Factual Explanation (Beginner)

```
Prediction: Diabetic
Calibrated Probability: 0.468

Factors impacting the probability for class Diabetic positively:
* Glucose (145.0) > 139.50
* Age (57.0) > 27.50
* Pregnancies (13.0) > 6.50

Factors impacting the probability for class Diabetic negatively:
* BMI (22.2) <= 26.90
```

## Instance 1

### Factual Explanation (Advanced)

```
Prediction: Diabetic
Calibrated Probability: 0.468
Prediction Interval: [0.457, 0.4

In [8]:
# 2. Text - returns formatted text string
text_narratives = explanations.to_narrative(
    expertise_level="beginner", output_format="text"
)
print(text_narratives)
text_narratives = explanations.add_conjunctions(max_rule_size=3).to_narrative(
    expertise_level="beginner", output_format="text"
)
print(text_narratives)


Instance 0

Factual Explanation (Beginner):
--------------------------------------------------------------------------------
Prediction: Diabetic
Calibrated Probability: 0.468

Factors impacting the probability for class Diabetic positively:
* Glucose (145.0) > 139.50
* Age (57.0) > 27.50
* Pregnancies (13.0) > 6.50

Factors impacting the probability for class Diabetic negatively:
* BMI (22.2) <= 26.90

Instance 1

Factual Explanation (Beginner):
--------------------------------------------------------------------------------
Prediction: Diabetic
Calibrated Probability: 0.468

Factors impacting the probability for class Diabetic positively:
* BMI (34.1) > 26.90
* SkinThickness (0.0) <= 32.50
* Age (38.0) > 27.50 [⚠️ direction uncertain]

Factors impacting the probability for class Diabetic negatively:
* Glucose (117.0) <= 139.50
* BloodPressure (92.0) > 63.00
* DiabetesPedigreeFunction (0.34) <= 0.56 [⚠️ direction uncertain]

Instance 2

Factual Explanation (Beginner):
---------------

In [9]:
# 3. HTML - returns HTML table
html_narratives = explanations.to_narrative(
    expertise_level=("beginner", "intermediate", "advanced"),
    output_format="html",
)
from IPython.display import HTML

HTML(html_narratives)

In [10]:
# 4. Dictionary - returns list of dictionaries
dict_narratives = explanations.to_narrative(
    expertise_level="advanced", output_format="dict"
)
dict_narratives

[{'instance_index': 0,
  'factual_explanation_advanced': 'Prediction: Diabetic\nCalibrated Probability: 0.468\nPrediction Interval: [0.457, 0.478]\n\nFactors impacting the calibrated probability for class Diabetic positively:\nGlucose (145.0) > 139.50                                                         — weight ≈ 0.278 [0.273, 0.297]\nAge (57.0) > 27.50                                                               — weight ≈ 0.069 [0.045, 0.103]\nPregnancies (13.0) > 6.50                                                        — weight ≈ 0.050 [0.019, 0.091]\n\nFactors impacting the calibrated probability for class Diabetic negatively:\n(Pregnancies (13.0) > 6.50 AND Glucose (145.0) > 139.50 AND Age (57.0) > 27.50)  — weight ≈ -0.347 [-0.371, -0.343]\n(Pregnancies (13.0) > 6.50 AND Glucose (145.0) > 139.50)                         — weight ≈ -0.288 [-0.311, -0.283]\n(Pregnancies (13.0) > 6.50 AND Glucose (145.0) > 139.50 AND BMI (22.2) <= 26.90) — weight ≈ -0.288 [-0.310, -0.283]\n(

In [11]:
# Different expertise levels:

# Single level
beginner_only = explanations.to_narrative(
    expertise_level="beginner", output_format="text"
)
print(beginner_only)


Instance 0

Factual Explanation (Beginner):
--------------------------------------------------------------------------------
Prediction: Diabetic
Calibrated Probability: 0.468

Factors impacting the probability for class Diabetic positively:
* Glucose (145.0) > 139.50
* Age (57.0) > 27.50
* Pregnancies (13.0) > 6.50

Factors impacting the probability for class Diabetic negatively:
* (Pregnancies (13.0) > 6.50 AND Glucose (145.0) > 139.50 AND Age (57.0) > 27.50)
* (Pregnancies (13.0) > 6.50 AND Glucose (145.0) > 139.50)
* (Pregnancies (13.0) > 6.50 AND Glucose (145.0) > 139.50 AND BMI (22.2) <= 26.90)
* (Glucose (145.0) > 139.50 AND BMI (22.2) <= 26.90)
* (BMI (22.2) <= 26.90 AND Glucose (145.0) > 139.50 AND Age (57.0) > 27.50)
* (Glucose (145.0) > 139.50 AND Age (57.0) > 27.50)
* BMI (22.2) <= 26.90
* (Pregnancies (13.0) > 6.50 AND Age (57.0) > 27.50)
* (Pregnancies (13.0) > 6.50 AND BMI (22.2) <= 26.90)
* (BMI (22.2) <= 26.90 AND Age (57.0) > 27.50) [⚠️ direction uncertain]

Instance

In [12]:
# Multiple levels
all_levels = explanations.to_narrative(
    expertise_level=("beginner", "intermediate", "advanced"),
    output_format="dataframe",
)
all_levels

,instance_index,factual_explanation_beginner,factual_explanation_intermediate,factual_explanation_advanced,expertise_level,problem_type
0,0,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, intermediate, advanced)",binary_classification
1,1,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, intermediate, advanced)",binary_classification
2,2,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, intermediate, advanced)",binary_classification
3,3,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, intermediate, advanced)",binary_classification
4,4,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, intermediate, advanced)",binary_classification
5,5,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, intermediate, advanced)",binary_classification
6,6,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, intermediate, advanced)",binary_classification
7,7,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, intermediate, advanced)",binary_classification
8,8,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, intermediate, advanced)",binary_classification
9,9,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, intermediate, advanced)",binary_classification


In [13]:
# Template path handling:

# If exp.yaml doesn't exist, automatically falls back to explain_template.yaml
narratives = explanations.to_narrative(
    template_path="exp.yaml",  # Will use default if not found
    expertise_level=("beginner", "advanced"),
    output_format="dataframe",
)
narratives

C:\Users\loftuw\Documents\Github\moffran\calibrated_explanations\src\calibrated_explanations\explanations\explanations.py:1507: UserWarning: Narrative template fallback: default template used because provided relative path was missing
  return plugin.plot(


,instance_index,factual_explanation_beginner,factual_explanation_advanced,expertise_level,problem_type
0,0,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, advanced)",binary_classification
1,1,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, advanced)",binary_classification
2,2,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, advanced)",binary_classification
3,3,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, advanced)",binary_classification
4,4,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, advanced)",binary_classification
5,5,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, advanced)",binary_classification
6,6,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, advanced)",binary_classification
7,7,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, advanced)",binary_classification
8,8,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, advanced)",binary_classification
9,9,Prediction: Diabetic\nCalibrated Probability: ...,Prediction: Diabetic\nCalibrated Probability: ...,"(beginner, advanced)",binary_classification


In [14]:
# Use custom template
narratives = explanations.to_narrative(
    template_path="/path/to/custom_template.yaml",
    expertise_level=("beginner", "advanced"),
    output_format="dataframe",
)

# Use default template
narratives = explanations.to_narrative(
    expertise_level=("beginner", "advanced"), output_format="dataframe"
)

print("The to_narrative method provides a clean, intuitive API for generating narratives!")

The to_narrative method provides a clean, intuitive API for generating narratives!


C:\Users\loftuw\Documents\Github\moffran\calibrated_explanations\src\calibrated_explanations\explanations\explanations.py:1507: UserWarning: Narrative template fallback: default template used because provided relative path was missing
  return plugin.plot(
